In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# 1. Cargar los tres archivos CSV
# =========================================================
df_long = pd.read_csv('../outputs/collahuasi_wells_cleaned_long.csv')
df_meta = pd.read_csv('../data/datos_collahuasi/tabla_8_4_punteras_limpia_2.csv')
df_niveles = pd.read_csv('../data/datos_collahuasi/tabla_8_5_niveles_limpia.csv')

# =========================================================
# 2. Metadatos de Pozos (CSV 2: tabla_8_4_punteras_limpia_2)
# =========================================================
df_meta_clean = df_meta[['id', 'BHID', 'este_m', 'norte_m', 'elevacion_msnm', 'prof_m']].copy()
df_meta_clean.rename(columns={
    'BHID': 'name',
    'este_m': 'east_m',
    'norte_m': 'north_m',
    'elevacion_msnm': 'elevation_masl',
    'prof_m': 'total_depth_m'
}, inplace=True)

# Limpiar identificadores y convertir campos numéricos
df_meta_clean['id'] = df_meta_clean['id'].astype(str).str.strip()
df_meta_clean['name'] = df_meta_clean['name'].astype(str).str.strip()

for col in ['east_m', 'north_m', 'elevation_masl', 'total_depth_m']:
    df_meta_clean[col] = pd.to_numeric(
        df_meta_clean[col].astype(str).str.replace('-', ''), 
        errors='coerce'
    )

# =========================================================
# 3. Observaciones CSV 1 (collahuasi_wells_cleaned_long)
# =========================================================
df_obs1 = df_long[['BHID', 'DATE', 'WATER_LEVEL', 'OBS']].copy()
df_obs1.rename(columns={
    'BHID': 'name',
    'DATE': 'date',
    'WATER_LEVEL': 'level_mbgs',
    'OBS': 'observations'
}, inplace=True)

df_obs1['name'] = df_obs1['name'].astype(str).str.strip()
df_obs1['date'] = pd.to_datetime(df_obs1['date'], errors='coerce')
df_obs1['level_mbgs'] = pd.to_numeric(df_obs1['level_mbgs'], errors='coerce')
df_obs1['level_masl'] = np.nan

# =========================================================
# 4. Observaciones CSV 3 (tabla_8_5_niveles_limpia - Unpivot a formato largo)
# =========================================================
date_mapping = [
    ('prof_ene_04_mbnt',     'cota_ene_04_msnm',     '2004-01-01'),
    ('prof_15_feb_04_mbnt',  'cota_15_feb_04_msnm',  '2004-02-15'),
    ('prof_29_feb_04_mbnt',  'cota_29_feb_04_msnm',  '2004-02-29'),
    ('prof_abr_07_mbnt',     'cota_abr_07_msnm',     '2007-04-01'),
    ('prof_jun_07_mbnt',     'cota_jun_07_msnm',     '2007-06-01'),
    ('prof_ene_08_mbnt',     'cota_ene_08_msnm',     '2008-01-01')
]

dfs_periodos = []
for prof_col, cota_col, fecha_str in date_mapping:
    if prof_col in df_niveles.columns and cota_col in df_niveles.columns:
        temp_df = df_niveles[['id', prof_col, cota_col]].copy()
        temp_df.rename(columns={
            prof_col: 'level_mbgs',
            cota_col: 'level_masl'
        }, inplace=True)
        temp_df['date'] = pd.to_datetime(fecha_str)
        dfs_periodos.append(temp_df)

df_obs2 = pd.concat(dfs_periodos, ignore_index=True)
df_obs2['id'] = df_obs2['id'].astype(str).str.strip()

# Convertir columnas a numérico (removiendo posibles '-' o espacios)
df_obs2['level_mbgs'] = pd.to_numeric(df_obs2['level_mbgs'].astype(str).str.replace('-', ''), errors='coerce')
df_obs2['level_masl'] = pd.to_numeric(df_obs2['level_masl'].astype(str).str.replace('-', ''), errors='coerce')

# Descartar filas donde no haya ni profundidad ni cota registrada
df_obs2 = df_obs2[df_obs2['level_mbgs'].notna() | df_obs2['level_masl'].notna()].copy()
#df_obs2['OBS'] = np.nan

# =========================================================
# 5. Cruzar identificadores y unir series temporales
# =========================================================
# Asignar 'id' a df_obs1 y 'name' a df_obs2 a partir de los metadatos
df_obs1 = df_obs1.merge(df_meta_clean[['name', 'id']], on='name', how='left')
df_obs2 = df_obs2.merge(df_meta_clean[['id', 'name']], on='id', how='left')

# Concatenar ambas series temporales
df_obs_total = pd.concat([df_obs1, df_obs2], ignore_index=True)

# Deduplicación por (pozo, fecha) priorizando registros que tengan level_masl y OBS
df_obs_total = df_obs_total.sort_values(by=['name', 'date', 'level_masl'], na_position='first')
df_obs_total = df_obs_total.drop_duplicates(subset=['name', 'date'], keep='last')

# =========================================================
# 6. Merge final con Metadatos (conserva pozos sin mediciones)
# =========================================================
df_final = df_obs_total.merge(df_meta_clean, on=['id', 'name'], how='outer')

# Si 'level_masl' está vacío, calcularlo a partir de: elevation_masl - level_mbgs
df_final['level_masl'] = df_final['level_masl'].combine_first(
    df_final['elevation_masl'] - df_final['level_mbgs']
)

# =========================================================
# 7. Ordenar columnas finales (OBS al final) y exportar
# =========================================================
column_order = [
    'id',
    'name',
    'date',
    'level_mbgs',
    'level_masl',
    'east_m',
    'north_m',
    'elevation_masl',
    'total_depth_m',
    'observations'  # <--- Columna OBS como última columna
]

df_final = df_final[column_order].sort_values(by=['id', 'date'], na_position='first').reset_index(drop=True)

# Guardar resultado consolidado
df_final.to_csv('../outputs/consolidated_collahuasi_wells.csv', index=False)
print(f"Dataset generado exitosamente con {len(df_final)} registros y {len(df_final.columns)} columnas.")


Dataset generado exitosamente con 2681 registros y 10 columnas.
